# Bitacoras onomásticas SEMEFO 1968-1982

In [1]:
import pandas as pd
from numpy import nan
from helper_pandas import simplify_string, df_to_dict

In [2]:
df = pd.read_pickle('data_clean/SEMEFO_BO_all.pkl')
df = df[['ID','Diagnostico']] 
df

,ID,Diagnostico
0,BO_1968_00001,S-D
1,BO_1968_00002,S-D
2,BO_1968_00003,S-D
3,BO_1968_00004,S-D
4,BO_1968_00005,S-D
...,...,...
96855,BO_1982_07489,S-D
96856,BO_1982_07490,S-D
96857,BO_1982_07491,S-D
96858,BO_1982_07492,S-D


In [3]:
data = df['Diagnostico'].value_counts().reset_index()
data

,Diagnostico,count
0,S-D,55894
1,TM,7505
2,TCE,5939
3,BN,1826
4,CVG,1287
...,...,...
3712,HC+TMI,1
3713,HC+Tb,1
3714,HC+Tumor,1
3715,HCC+A,1


# Catálogo diagnostico

## Remplazar especiales

In [4]:
cat_replace = pd.read_csv('catalogos/catalogo_diagnostico_remplazo.csv')
cat_replace = df_to_dict(cat_replace)
cat_replace
df['Diagnostico'] = df['Diagnostico'].replace(cat_replace)
df.head()

,ID,Diagnostico
0,BO_1968_00001,S-D
1,BO_1968_00002,S-D
2,BO_1968_00003,S-D
3,BO_1968_00004,S-D
4,BO_1968_00005,S-D


## Separar y simplificar

In [5]:
def break_and_simplify(string, sort=False):
    string = string.replace(' x ','+').replace(' y ','+')
    string = string.split('+')
    string = [simplify_string(s, str_case=None) for s in string]
    if sort: string = sorted(string, sort=True)
    return string

sep = [break_and_simplify(s) for s in data['Diagnostico'].to_list()]
sep = pd.DataFrame(sep)
sep.columns = [f'Sep{i}' for i in sep.columns]
sep

,Sep0,Sep1,Sep2,Sep3,Sep4
0,S-D,None,None,None,None
1,TM,None,None,None,None
2,TCE,None,None,None,None
3,BN,None,None,None,None
4,CVG,None,None,None,None
...,...,...,...,...,...
3712,HC,TMI,None,None,None
3713,HC,Tb,None,None,None
3714,HC,Tumor,None,None,None
3715,HCC,A,None,None,None


In [6]:
cat_simple = pd.read_csv('catalogos/catalogo_diagnostico_simplificado.csv')
cat_simple = df_to_dict(cat_simple)
cat_simple

simple = pd.DataFrame(columns=sep.columns)
for col in sep.columns:
    simple[col] = sep[col].map(cat_simple)
simple

,Sep0,Sep1,Sep2,Sep3,Sep4
0,S-D,NaN,NaN,NaN,NaN
1,TM,NaN,NaN,NaN,NaN
2,TCE,NaN,NaN,NaN,NaN
3,BN,NaN,NaN,NaN,NaN
4,CVG,NaN,NaN,NaN,NaN
...,...,...,...,...,...
3712,HC,TMI,NaN,NaN,NaN
3713,HC,TB,NaN,NaN,NaN
3714,HC,NaN,NaN,NaN,NaN
3715,NaN,A,NaN,NaN,NaN


In [7]:
missing = sep.where( sep.isna()!=simple.isna() )
missing

,Sep0,Sep1,Sep2,Sep3,Sep4
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
3712,NaN,NaN,NaN,NaN,NaN
3713,NaN,NaN,NaN,NaN,NaN
3714,NaN,Tumor,NaN,NaN,NaN
3715,HCC,NaN,NaN,NaN,NaN


In [15]:
missing['Miss_sum'] =(sep.isna()!=simple.isna()).sum(axis=1)

In [17]:
simple.columns = [f'Simple{i}' for i in range(simple.shape[1])]
missing.columns = [f'Miss{i}' for i in range(missing.shape[1]-1)] + ['Miss_sum']

data = pd.concat([data, sep, simple, missing], axis=1)
data.to_csv('data_raw/diagnostico_separados.csv', index=False)
data

,Diagnostico,count,Sep0,Sep1,Sep2,Sep3,Sep4,Simple0,Simple1,Simple2,Simple3,Simple4,Miss0,Miss1,Miss2,Miss3,Miss4,Miss_sum
0,S-D,55894,S-D,None,None,None,None,S-D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,TM,7505,TM,None,None,None,None,TM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,TCE,5939,TCE,None,None,None,None,TCE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,BN,1826,BN,None,None,None,None,BN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,CVG,1287,CVG,None,None,None,None,CVG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3712,HC+TMI,1,HC,TMI,None,None,None,HC,TMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3713,HC+Tb,1,HC,Tb,None,None,None,HC,TB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3714,HC+Tumor,1,HC,Tumor,None,None,None,HC,NaN,NaN,NaN,NaN,NaN,Tumor,NaN,NaN,NaN,1
3715,HCC+A,1,HCC,A,None,None,None,NaN,A,NaN,NaN,NaN,HCC,NaN,NaN,NaN,NaN,1


In [23]:
pd.concat( [missing[f'Miss{i}'] for i in range(missing.shape[1]-1)] ).value_counts().sort_index()

                            63
"No Exp"                     1
"TETANOS"                    1
"Tetanos"                    1
(Aborto) A Septico           1
                            ..
quemaduras                   1
rup de art Pulmonar no T     1
sd                           1
shock                        1
trom                         1
Name: count, Length: 2320, dtype: int64